# 🚀 Getting Started with the OSPool

Welcome! This notebook walks through the basics on how to use the **Open Science Pool (OSPool)** — a national computing resource that brings together contributions from institutions across the United States. The OSPool is operated by the [OSG Consortium](https://osg-htc.org) through the [PATh project](https://path-cc.io), and it uses [HTCondor](https://htcondor.org) to manage jobs.

## About this environment
This notebook is powered by a mini-HTCondor pool running inside your Jupyter session and behaves like an OSPool Access Point (AP), **but isn't the real OSPool**. Use this pseudo-AP to: 

* Test-drive HTCondor and the OSPool and learn more about how they work.
* Explore whether this is a good solution for your computational work.
* Start prototyping workloads for the OSPool.

> **Important**: All files created or changed in this notebook will be lost after the session closes!! If you do some work you want to keep, make sure to download the files you want before exiting. 

## Overview of the OSPool

Below is a diagram that models how submitting jobs to the OSPool works.

<p style="text-align: center"><img src="https://portal.osg-htc.org/documentation/assets/overview_htcondor_job_submission.png" width=700px></p>

- You get an account on the **Access Point (AP)**. This is where you log in and prepare your software, files, and scripts.
- **HTCondor** is the job scheduling software that manages the jobs (a unit of work) you submit.
- Your jobs run on separate machines we call **Execution Points (EP)**. When they finish, outputs are sent back to you at the AP.

## Typical workflow

1. **Log in** to an OSPool *Access Point (AP)* — for example `ap40.uw.osg-htc.org`. You're logged in through a notebook right now.
2. **Upload or write your files** that you need to run your work, including scripts, data, and software.
4. **Describe your work** in an HTCondor *submit file*.
5. **Submit your work** with `condor_submit`, watch it with `condor_q`, and review your work after it runs.

You can explore each of these concepts through the next section. 

# 👩🏽‍💻 Hands-on Overview

Each grey cell below is a shell command. Click a cell and press **Shift+Enter** to run it.

## 0. Check that the pool is running

Everything in this notebook talks to an HTCondor pool. Let's make sure ours is up (and start it if it isn't).

In [ ]:
# Ask the pool for a summary of its resources. If it doesn't answer, start the local pool and give it a few seconds.
condor_status -total 2>/dev/null || { echo "Pool not responding - starting the local HTCondor pool..."; condor_master && sleep 10 && condor_status -total; }

## 1. Look around the Access Point

Let's look around the AP.

1. Check the version of HTCondor.

In [ ]:
condor_version

2. What machines/slots are available to run jobs?

   On the OSPool this lists many thousands of slots; our practice pool has just one (partitionable) slot on this machine.

In [ ]:
condor_status

3. What's in *your* job queue? (Empty for now.)

In [ ]:
condor_q

## 2. Describe our jobs

Let's run some work! For this notebook, we are going to create a job that uses a Python script to count words in text files. 

> **Think about your own work**: While this example specifically uses Python and `.txt` input files, the principles apply to any script or code that needs to be run on many different inputs -- either files or arguments. 

### Our script: `wordcount.py`

Let's test our script. It should give us a summary of the word count task.

In [ ]:
./wordcount.py Alice_in_Wonderland.txt

It also creates a `counts.Alice_in_Wonderland.tsv` file with a detailed table. Let's check the first few lines of that file.

In [ ]:
head counts.Alice_in_Wonderland.tsv

> Normally, we would **not** test this script on the AP, because most researcher jobs are computationally heavy! However, our word count example is light, so we are testing it here on the AP to show you what we expect for our outputs.

Now, let's imagine we want to run this script for all the books we have:
- Alice_in_Wonderland.txt
- Huckleberry_Finn.txt
- Pride_and_Prejudice.txt

Instead of running this one-by-one, we're going to leverage the OSPool and *submit all of this work at once*! Let's list all these books into a text file called `book.list` — we will use this later!

In [ ]:
ls *.txt > book.list && cat book.list

Next, we will need a **submit file**, which translates the work we want to run into format that HTCondor understands.

### The submit file

Let's take a look at the submit file.

In [ ]:
cat wordcount.sub

### Submit file options

In our submit file, we use the following to describe our work.

| Submit file option | Purpose | Notes |
| --- | --- | --- |
| `shell` | The command you want to run. | More complex commands/scripts should be written in a wrapper script. |
| `container_image` | The container image that has our software environment installed. | For now, we're using a prebuilt container with Python installed. You can build and use your own containers on the OSPool. |
| `log` | A file created by HTCondor to record information about the job. |
| `error` | Where to save standard error messages. |
| `output` | Where to save standard output messages. |
| `transfer_input_files` | Files that we need to run our work. | The AP and the EPs that run your jobs do **not** share a filesystem, so list every file your job needs. |
| `request_*` | The resources (CPUs, memory, disk) we need to run our job. | GPUs may also be specified. |
| `queue` | How many jobs we want to run. Always at the end of the submit file. | This example uses the `queue <var> from <list>` syntax. |

> For more information about variables (i.e., `$(Cluster)`, `$(Process)`), see [Appendix](#Appendix).

## 3. Submit our jobs

Our submit file is ready to go! Let's submit the jobs:

In [ ]:
condor_submit wordcount.sub

## 4. Get the status of our jobs

Let's query the status of our job(s):

In [ ]:
condor_q

You can also try the `-nobatch` option to see one job per line.

In [ ]:
condor_q -nobatch

The jobs should complete within a few minutes!

> If you want a more "real-time" update of your jobs' status, open a Terminal tab, and use the command `condor_watch_q`. (It doesn't play nicely in the notebook interface, so don't run it in a notebook cell.)

## 5. Look at the results

When a job finishes, it leaves the queue, and `condor_q` will be empty. Let's check `condor_history` for a record of our jobs.

In [ ]:
condor_history -limit 5

If our job ran successfully, we should have three `.tsv` files (one for each book) and three `.out` files with a printed summary. Let's run the commands below to confirm.

In [ ]:
ls -lh counts.*.tsv

In [ ]:
cat out/*.out

If everything ran smoothly, the standard error files (specified by `error` in the submit file) should be empty. Let's check.

In [ ]:
cat err/*.err

The `log` file is HTCondor's own record of the job's life — submitted, matched, executing, terminated, plus resource usage. The very end of this file shows a table about resources requested, allocated, and used. **This is useful for testing resource usage!** We recommend testing one or twojobs first and tailoring resource requests before scaling up to a full workload.

In [ ]:
tail -n 20 log/*log

## 7. Where to go next

You just did the whole loop: write, submit, monitor, collect, scale. On the OSPool the steps are identical — only the size of the pool changes.

Some ideas for next steps: 
* **Explore additional examples:** [Tutorial Selection](select-tutorial.ipynb) -- Click the "run" button three times to view tutorial options. 
* **Get an OSPool account:** 
    * Have you already requested an account through our form? 
        * Schedule a meeting: [Consultation Booking](https://osgfacilitation.setmore.com/)
        * Come to office hours: [OSPool office hours](https://portal.osg-htc.org/documentation/support_and_training/support/getting-help-from-RCFs/)
    * Still need to request an account? Go here: [portal.osg-htc.org/application](https://portal.osg-htc.org/application)
* **Get help:** Email [support@osg-htc.org](mailto:support@osg-htc.org) or drop by [OSPool office hours](https://portal.osg-htc.org/documentation/support_and_training/support/getting-help-from-RCFs/)

Additional reference material to explore: 
* **OSPool documentation:** [portal.osg-htc.org/documentation](https://portal.osg-htc.org/documentation/) — start with the [Quickstart](https://portal.osg-htc.org/documentation/htc_workloads/submitting_workloads/tutorial-quickstart/) and the [Roadmap to HTC Workload Submission](https://portal.osg-htc.org/documentation/htc_workloads/workload_planning/roadmap/)
* **HTCondor manual:** [htcondor.readthedocs.io](https://htcondor.readthedocs.io/)

## 8. Clean up (optional)

Remove the files this notebook created so you can run it again from a clean slate.

In [ ]:
rm -rf log err out counts.*.tsv book.list

# 📚 Appendix

### Variables: `$(Cluster)`, `$(Process)`, and more

Any time you see this `$(variable)` syntax, this is HTCondor's variable syntax. Some have default values, like `$(Cluster)` and `$(Process)`, but you can also define your own.

| Variables | Purpose |
| --- | --- |
| `$(Cluster)` | An automatically generated unique ID per submission. |
| `$(Process)` | An automatically generated ID for jobs in each submission. i.e., if one submission has 3 jobs, `$(Process)` goes from 0-2. |
| `$(book)` | A custom variable we made for this example. |

### Where is "Alice_in_Wonderland.txt" in the submit file?

Recall the `book.list` file we made — this lists "Alice_in_Wonderland.txt", as well as other books we want to analyze.

In the `queue` statement, we used the `queue <var> from <list>` syntax.

What this does:
1. HTCondor will look for the `book.list` file.
2. HTCondor reads each line and assign its value to the variable `$(book)`. Each line corresponds to a unique job.
3. In the submit file, HTCondor replaces `$(book)` with its value for that job.

Essentially, HTCondor uses and handles a "for" loop to submit multiple jobs.

**As a result, we're able to analyze multiple books using one submit command and file instead of having to write many submit files!**